## Rag Pipeline: Data Ingestion to Vector DB Pipeline ##

In [2]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
## Reading PDF files from a directory

def process_pdf_files(pdf_directory):
    pdf_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in the directory: {pdf_directory}")
    
    for pdf_file in pdf_files:
        loader = PyMuPDFLoader(str(pdf_file))
        documents = loader.load()

        #Add source information to metadata
        for doc in documents:
            doc.metadata["source_file"] = pdf_file.name
            doc.metadata["file_type"] = 'pdf'

        pdf_documents.extend(documents)

        print(f" Loaded {len(documents)} pages from {pdf_file.name}")
    return pdf_documents



all_pdf_documents = process_pdf_files("../data/pdf_files")

Found 1 PDF files in the directory: ../data/pdf_files
 Loaded 27 pages from 2026 Pinnacle Commissions_TBM AS KAM_GI PRIMA.pdf


## Chunking  (Text splitting)

In [17]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200): 
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,length_function=len, separators=["\n\n", "\n", " ", ""])
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")


    #show example of the first chunk
    if split_docs:
        print("Example of the first chunk:")
        print(split_docs[1].page_content)  # Print the first 500 characters of the first chunk
        print("Metadata:", split_docs[1].metadata)


    return split_docs


In [18]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 27 documents into 48 chunks.
Example of the first chunk:
Proprietary and confidential — do not distribute
Dear Colleagues,
I want to begin by congratulating each one of you for your relentless commitment and determination in 
driving Abbott India Limited’s positive margins and growth throughout the year. Despite the challenges 
we faced, your perseverance and focus on Gross Margin Improvement have been truly commendable.
As we step into 2026, our collective priority must be consistent target achievement. To support you 
in this journey, we are excited to introduce the 2026 Pinnacle Commissions Program, designed to 
make earning opportunities simpler, more rewarding, and aligned with your efforts.
Key Highlights of 2026 Pinnacle Commissions Program:
1. Focus on strategic objectives: Components of monthly commission and league table now include 
strategic objectives for our focus brands and well as Business Units
2. Consistency is key: Quarter commissions will be basis consecutive 

## Embedding and VectorStoreDB

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        """Initialize the EmbeddingManager with a specified sentence transformer model.

        Args:
            model_name (str, optional): The name of the sentence transformer model to use for embeddings. Defaults to "all-MiniLM-L6-v2".
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model: {self.model_name}, Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        """

        if not self.model:
            raise ValueError("Model is not loaded. Please check the model name or installation.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## Initialize the EmbeddingManager and generate embeddings for the chunks
embedding_manager = EmbeddingManager(model_name="all-MiniLM-L6-v2")
embedding_manager

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\PAGARAX1\AppData\Local\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PAGARAX1\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: all-MiniLM-L6-v2, Embedding dimension: 384


## Vector Store using ChromaDB

In [23]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the VectorStore with a specified collection name.

        Args:
            collection_name (str, optional): The name of the ChromaDB collection to use. Defaults to "pdf_documents".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        """Initialize the ChromaDB client and collection."""
        try:
            #Create persistent chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG project"}
                )
            
            print(f"Initialized ChromaDB collection: {self.collection_name} at {self.persist_directory}")
            print(f"Existing documents in the collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing ChromaDB collection: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the ChromaDB collection.

        Args:
            documents (List[Any]): A list of LangChain Document objects to add to the collection.
            embeddings (np.ndarray): A 2D numpy array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the collection: {self.collection_name}")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            #Generate a unique ID for each document
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)

            #Prepare metadata 
            metadata = dict(doc.metadata)  # Ensure metadata is a dictionary
            metadata["doc_index"] = i  # Add index to metadata
            metadata["context_length"] = len(doc.page_content)  # Add context length to metadata
            metadatas.append(metadata)
           
            #Store the document text and embedding
            documents_texts.append(doc.page_content)
            embeddings_list.append(emb)

        #Add to ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
            )
            print(f"Added {len(documents)} documents to the collection: {self.collection_name}")
            print(f"Total documents in the collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to ChromaDB collection: {e}")
            raise

vector_store = VectorStore()
vector_store


Initialized ChromaDB collection: pdf_documents at ../data/vector_store
Existing documents in the collection: 0


In [27]:
#Converting the text to embeddings

texts = [doc.page_content for doc in chunks]


#Generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

#store the documents and embeddings in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 48 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (48, 384)
Adding 48 documents to the collection: pdf_documents
Added 48 documents to the collection: pdf_documents
Total documents in the collection after addition: 48


## Retrieval